# Analysis of DM Density Dependence on Theta1 and Vheta1

## Imports

In [ ]:
# Important modules
import numpy as np
import itertools
import matplotlib.pyplot as plt
import json

# Kinetic misalignment functions
from kin_mis_utils import evolution_kinetic_mis

# Some plot settings
import matplotlib as mpl
mpl.rcParams['mathtext.rm'] = 'serif'
mpl.rcParams['mathtext.fontset'] = 'cm'
mpl.rcParams['font.family'] = 'serif'
mpl.rcParams['font.size'] = 16
mpl.rcParams['text.usetex'] = True

## Parameters

In [ ]:
tautab = np.linspace(1, 5, 1000) # Normalized conformal time array, SHOULD START AT 1!
fAGeV = 1e10 # Axion decay constant in GeV

## Varying Initial Velocity

In [ ]:
theta1 = 0
vheta1_array = np.logspace(0, 1, 5)
density_vheta = []

for vheta1 in vheta1_array:
    result = evolution_kinetic_mis(fAGeV, theta1, vheta1, tautab)
    omega_avg = np.mean(result['Omegah2tab'][-3:]) # Average over last 3 values
    density_vheta.append(omega_avg)

density_vheta = np.array(density_vheta)

In [ ]:
plt.figure(figsize = (8, 6))
plt.plot(vheta1_array, density_vheta, marker = '.', linestyle = 'None')
plt.xscale('log')
plt.yscale('log')
plt.xlabel(r'$\dot{\Theta}_1/H_1$')
plt.ylabel(r'$\Omega_a h^2$')
plt.title('Relic Density vs. Initial Velocity')
plt.axhline(0.12, color = 'black', linestyle = '--')
textstr = r'$f_a = {:.2e} \ \mathrm{{GeV}}$'.format(fAGeV) + '\n' + r'${{\Theta}}_1 = {:.2f}$'.format(theta1)
plt.gca().text(0.05, 0.90, textstr, transform = plt.gca().transAxes,
               fontsize = 14, verticalalignment = 'top', linespacing = 1.8, fontweight = 'bold',
               bbox = dict(boxstyle = 'round', facecolor = 'white', alpha = 0.8))
plt.tight_layout()
plt.show()
plt.close()

## Varying Initial Field Value

In [ ]:
theta1_array = np.linspace(0, 2*np.pi, 10)
vheta1 = 50.0
density_theta = []

for theta1 in theta1_array:
    result = evolution_kinetic_mis(fAGeV, theta1, vheta1, tautab)
    omega_avg = np.mean(result['Omegah2tab'][-3:]) # Average over last 3 values
    density_theta.append(omega_avg)

density_theta = np.array(density_theta)

In [ ]:
plt.figure(figsize = (8, 6))
plt.plot(theta1_array, density_theta, marker = '.', linestyle = 'None')
plt.xscale('log')
plt.xlabel(r'$\Theta_1$')
plt.ylabel(r'$\Omega_a h^2$')
plt.title('Relic Density vs. Initial Misalignment Angle')
plt.axhline(0.12, color = 'black', linestyle = '--')
textstr = (fr'$f_a = {fAGeV:.2e} \ \mathrm{{GeV}}$' + '\n' +
           fr'$\dot{{\Theta}}_1/H_1 = {vheta1:.2f}$')
plt.gca().text(0.05, 0.90, textstr, transform = plt.gca().transAxes,
               fontsize = 14, verticalalignment = 'top', linespacing = 1.8, fontweight = 'bold',
               bbox = dict(boxstyle = 'round', facecolor = 'white', alpha = 0.8))
plt.tight_layout()
plt.show()
plt.close()

## Varying Initial Velocity and Field Value

In [ ]:
# Add dictionaries if needed
file_paths = ['parameter_matches_1e+12.json', 'parameter_matches_1e+13.json',
              'parameter_matches_1e+14.json', 'parameter_matches_1e+15.json'] # List of JSON file paths

if 0:
    # Step 1: Load each file into a dictionary
    dict_list = []
    for path in file_paths:
        with open(path, 'r') as f:
            data = json.load(f)
            data = {float(k): v for k, v in data.items()}
            dict_list.append(data)
    
    # Step 2: Combine dictionaries
    combined = defaultdict(list)
    for d in dict_list:
        for key, val in d.items():
            combined[key].extend(val)
    
    # Step 3: Sort each list of values by the second element (val[1])
    for key in combined:
        combined[key] = sorted(combined[key], key = lambda x: x[1])
    
    # Step 4: Save the sorted combined dictionary to a new JSON file
    combined_serializable = {str(k): v for k, v in combined.items()}
    
    with open('parameter_matches.json', 'w') as f:
        json.dump(combined_serializable, f)

In [ ]:
# Read JSON file into a Python dict
with open('parameter_matches.json', 'r') as f:
     param_pairs_dict = json.load(f)

# Convert keys to float
param_pairs_dict = {float(k): v for k, v in param_pairs_dict.items()}

In [ ]:
plt.figure(figsize = (12, 8))

for k, v in param_pairs_dict.items():
    theta1_vals = np.asarray([t[0] for t in v])
    vheta1_vals = np.asarray([t[1] for t in v])

    plt.plot(vheta1_vals, theta1_vals, label = fr'$f_a = ${k:.0e}GeV', marker = '.', linestyle = '')
    
plt.xlabel(r'$\dot{\Theta}_1/H_1$')
plt.ylabel(r'$\Theta_1$')
plt.xscale('log')
plt.title('Correct Dark Matter Abundance', fontweight = 'bold')
plt.yticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi], ['0', r'$\frac{\pi}{2}$', r'$\pi$', r'$\frac{3\pi}{2}$', r'$2\pi$'])
plt.tight_layout()
plt.legend(loc = 'lower right')
plt.show()
plt.close()

In [ ]:
plt.figure(figsize = (12, 8))

markers = itertools.cycle(['o', 's', '^', 'D', 'P', '*', 'X', 'H', 'v']) # Marker styles

for k, v in param_pairs_dict.items():

    theta1_vals = np.asarray([t[0] for t in v])
    vheta1_vals = np.asarray([t[1] for t in v])
    theta_max_vals = np.asarray([t[2] for t in v])%(2*np.pi)

    marker = next(markers)

    sc = plt.scatter(vheta1_vals, theta1_vals, c = theta_max_vals, marker = marker, edgecolor = 'k',
                     s = 50, alpha = 0.8, label = fr'$f_a = {k:.0e}$ GeV', vmin = 0, vmax = np.pi)

plt.xlabel(r'$\dot{\Theta}_1/H_1$')
plt.ylabel(r'$\Theta_1$')
plt.xscale('log')
plt.title('Correct Dark Matter Abundance', fontweight = 'bold')

plt.yticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi], ['0', r'$\frac{\pi}{2}$', r'$\pi$', r'$\frac{3\pi}{2}$', r'$2\pi$'])

# Add colorbar for theta_max
cbar = plt.colorbar(sc)
cbar.set_label(r'$\Theta_{\mathrm{max}}$', rotation = 270, labelpad = 15)

plt.tight_layout()
plt.legend(loc = 'lower right')
plt.show()
plt.close()

In [ ]:
# Choose two velocity bin indices to analyze
selected_bin_index_1 = 2
selected_bin_index_2 = 8  

# Collect all vheta1 and theta_max values (mod 2π)
all_velocities = []
all_theta_max_mod = []

for v in param_pairs_dict.values():
    vheta1_vals = [t[1] for t in v]
    theta_max_vals = [t[2] % (2 * np.pi) for t in v]

    all_velocities.extend(vheta1_vals)
    all_theta_max_mod.extend(theta_max_vals)

all_velocities = np.array(all_velocities)
all_theta_max_mod = np.array(all_theta_max_mod)

# Filter valid data
valid_mask = (all_velocities > 0) & np.isfinite(all_velocities) & np.isfinite(all_theta_max_mod)
all_velocities = all_velocities[valid_mask]
all_theta_max_mod = all_theta_max_mod[valid_mask]

# Define velocity bins
velocity_bins = np.logspace(np.log10(min(all_velocities)), np.log10(max(all_velocities)), num = 10)

# Histogram bins for theta_max
theta_bins = np.linspace(0, 2 * np.pi, 50)
bin_centers = 0.5 * (theta_bins[1:] + theta_bins[:-1])

# Function to extract normalized histogram for a velocity bin
def get_theta_max_distribution(bin_index):
    mask = (all_velocities >= velocity_bins[bin_index]) & (all_velocities < velocity_bins[bin_index + 1])
    selected_theta_max = all_theta_max_mod[mask]
    if len(selected_theta_max) == 0:
        return None
    counts, _ = np.histogram(selected_theta_max, bins = theta_bins)
    counts_percent = counts / counts.sum() * 100
    return counts_percent

# Get distributions
dist1 = get_theta_max_distribution(selected_bin_index_1)
dist2 = get_theta_max_distribution(selected_bin_index_2)

# Plot
plt.figure(figsize = (10, 6))
plt.title(r'Distribution of $\Theta_{max}$', fontweight = 'bold')
plt.xlabel(r'$\Theta_{max}$')
plt.ylabel('Percentage')
plt.xlim(0, np.pi+0.1)

if dist1 is not None:
    plt.bar(bin_centers, dist1, width = bin_centers[1] - bin_centers[0],
            edgecolor = 'k', alpha = 0.6, label = f'$\\dot{{\\Theta_1}}/H_1 \\in $[{velocity_bins[selected_bin_index_1]:.1e}, {velocity_bins[selected_bin_index_1 + 1]:.1e}]')

if dist2 is not None:
    plt.bar(bin_centers, dist2, width = bin_centers[1] - bin_centers[0],
            edgecolor = 'k', alpha = 0.6, label = f'$\\dot{{\\Theta_1}}/H_1 \\in $[{velocity_bins[selected_bin_index_2]:.1e}, {velocity_bins[selected_bin_index_2 + 1]:.1e}]')

plt.xticks(
    [0, np.pi / 2, np.pi,],
    ['0', r'$\frac{\pi}{2}$', r'$\pi$']
)
plt.legend()
plt.tight_layout()
plt.show()